[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h04_hanging_chain.ipynb)

# Hang a chain, flip it, build an arch

A chain can only pull. Turn its hanging shape upside down and every pull becomes a push: an arch in which every
stone is squeezed and nothing bends. Robert Hooke hid this idea in an anagram in 1675, and Antoni Gaudí designed a
church with it, using cords and little sacks of lead. Here you can hang your own chain, measure how close it is to a
parabola, add a heavy weight and watch the chain kink, turn it over into an arch, and find how thin a round Roman
arch can be before it falls.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

def catenary_a(sag, half):
    """The chain y = a(cosh(x/a) − 1) that sags `sag` between nails at x = ±half (found by halving the range)."""
    lo, hi = 1e-4, 1e4
    for _ in range(200):
        a = (lo * hi) ** 0.5
        lo, hi = (a, hi) if a * (math.cosh(half / a) - 1) > sag else (lo, a)
    return a

def hanging(span, loads, n=400, sag=None):
    """A chain between nails at x = 0 and x = span: its own weight spread evenly, plus point loads [(where, weight)].
    Between loads it is straight; at each load the slope jumps. Returns x and the depth below the nails."""
    x = np.linspace(0, span, n + 1)
    w = np.full(n + 1, 1.0 / n)                     # the chain weighs 1 in total
    for where, weight in loads:
        w[int(round(where / span * n))] += weight
    left = (w[1:-1] * (span - x[1:-1])).sum() / span   # the left nail's share of the weight
    v = left - np.concatenate([[0], np.cumsum(w[1:-1])])
    y = np.concatenate([[0], np.cumsum(v * np.diff(x))])
    return x, y * (sag / y.max()) if sag else y

print("Ready. Run the next cell.")

## 1 · How close is the chain to a parabola?

Galileo wrote in 1638 that a hanging chain looks very like a parabola. Hang one between two nails a metre apart and
change how far it sags. The coral line is the chain; the sage line is the parabola through the same three points.

In [ ]:
def close(sag_cm=10):
    half, sag = 0.5, sag_cm / 100
    a = catenary_a(sag, half)
    x = np.linspace(-half, half, 801)
    chain = sag - a * (np.cosh(x / a) - 1)
    para = sag * (1 - (x / half) ** 2)
    gap = np.abs(chain - para).max()
    print(f"sag {sag_cm} cm: the chain and the parabola differ by at most {gap * 1000:.2f} mm")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(x, -chain, color=CORAL, lw=3, label="hanging chain")
    ax.plot(x, -para, color=SAGE, lw=1.5, label="parabola")
    ax.set_aspect("equal"); ax.set_xlabel("metres"); ax.legend(loc="lower right"); plt.show()

interact(close, sag_cm=IntSlider(value=10, min=1, max=150, description="sag (cm)"));

## 2 · One heavy weight

Tie a weight to the chain and slide it along. The chain kinks at the weight and runs almost straight on either
side: its shape depends on what it carries.

In [ ]:
def weight(where=0.3, heavy=2.0):
    x, y = hanging(1.0, [(where, heavy)], sag=0.4)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(x, -y, color=CORAL, lw=3)
    k = int(round(where * 400))
    ax.plot([x[k]], [-y[k] - 0.04], "s", color=CHALK, ms=6 + 4 * heavy)
    ax.set_aspect("equal"); ax.set_title(f"a weight {heavy:g} × the chain's own, at {where:.2f} m"); plt.show()

interact(weight, where=FloatSlider(value=0.3, min=0.05, max=0.95, step=0.05, description="where"),
         heavy=FloatSlider(value=2.0, min=0.0, max=10.0, step=0.5, description="weight"));

## 3 · Flip it

As hangs the flexible line, so, but inverted, will stand the rigid arch. Turn the chain over and lay stones along it:
every pull in the chain becomes a push in the arch.

In [ ]:
def flip(turn=1.0):
    x, y = hanging(1.0, [(0.3, 1.5)], sag=0.45)
    s = math.cos(math.pi * turn)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(x, -y * s, color=CORAL if turn < 0.5 else TEAL, lw=4)
    ax.set_ylim(-0.55, 0.55); ax.set_aspect("equal")
    ax.set_title("hanging: every link pulls" if turn < 0.5 else "turned over: every stone pushes"); plt.show()

interact(flip, turn=FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="turn over"));

## 4 · How thin can a round arch be?

A Roman arch is a semicircle, not the chain's curve. It stands if the chain's line, flipped (engineers call it the
line of thrust), fits somewhere inside its stone. Make the arch thinner and watch for the moment no line fits. The
thickness is a fraction of the arch's radius.

In [ ]:
def thrust(t, n=90, hs=1500):
    """Lines of thrust for a semicircular arch of thickness t (radius 1) carrying its own weight. For each crown thrust
    h, every joint gives a band the line must pass through; returns the line through the middle of the widest band."""
    ri, ro = 1 - t / 2, 1 + t / 2
    d = (math.pi / 2) / n
    k = (2 / 3) * (ro ** 3 - ri ** 3) / (ro ** 2 - ri ** 2) * math.sin(d / 2) / (d / 2)
    w = 0.5 * (ro ** 2 - ri ** 2) * d
    p = d * np.arange(1, n + 1)
    Wc = w * np.arange(1, n + 1)
    M = np.cumsum(w * k * np.sin(d * (np.arange(n) + 0.5)))
    h = Wc[-1] * np.arange(1, hs) / hs
    q = np.sin(p)[None, :] * Wc[None, :] + np.cos(p)[None, :] * h[:, None]
    lo = np.maximum(ri * h, (ri * q - M[None, :]).max(axis=1))
    hi = np.minimum(ro * h, (ro * q - M[None, :]).min(axis=1))
    if (hi - lo).max() < 0: return None
    j = int(np.argmax(hi - lo)); u = (lo[j] + hi[j]) / 2
    return np.concatenate([[0], p]), np.concatenate([[u / h[j]], (u + M) / q[j]])

def arch(thickness=0.30):
    t = thickness
    fig, ax = plt.subplots(figsize=(7, 4.2))
    th = np.linspace(-math.pi / 2, math.pi / 2, 200)
    for r in (1 - t / 2, 1 + t / 2): ax.plot(r * np.sin(th), r * np.cos(th), color=TEAL, lw=2)
    line = thrust(t)
    if line is None:
        ax.set_title(f"thickness {t:.3f} × radius: no line fits, the arch falls", color=CORAL)
    else:
        ang, rad = line
        ax.plot(np.concatenate([-(rad * np.sin(ang))[::-1], rad * np.sin(ang)]),
                np.concatenate([(rad * np.cos(ang))[::-1], rad * np.cos(ang)]), color=CORAL, lw=2.5)
        ax.set_title(f"thickness {t:.3f} × radius: it stands")
    ax.set_aspect("equal"); ax.set_ylim(-0.05, 1.25); plt.show()

interact(arch, thickness=FloatSlider(value=0.30, min=0.08, max=0.40, step=0.002, description="thickness"));

Now find the edge exactly: halve the range between a thickness that falls and one that stands, over and over.

In [ ]:
lo, hi = 0.05, 0.2
for _ in range(20):
    mid = (lo + hi) / 2
    lo, hi = (lo, mid) if thrust(mid) is not None else (mid, hi)
print(f"the thinnest semicircular arch: about {hi:.4f} × its radius (Milanković, 1907: 0.1075)")

## 5 · Check Hooke's anagram

In 1675 Hooke printed his answer scrambled into letters in alphabetical order. In 1705 his executor published the
Latin. Are they really the same letters? (In Hooke's day, u and v were one letter.)

In [ ]:
anagram = "abcccddeeeeefggiiiiiiiillmmmmnnnnnooprrsssttttttuuuuuuuux"
latin = "ut pendet continuum flexile sic stabit contiguum rigidum inversum"
letters = latin.replace(" ", "").replace("v", "u")
print(len(anagram), "letters;", "the same letters, the same counts" if Counter(anagram) == Counter(letters) else "different!")
print("sorted Latin:", "".join(sorted(letters)))

## 6 · The challenge from the video

Hang a light chain between two nails and tie one heavy weight to its middle. What shape does the chain make? And
turned upside down, what kind of arch is that? Think first, then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("A light chain, one heavy weight at its middle. What shape?   (press the button when you're ready)"); return
    x, y = hanging(1.0, [(0.5, 20.0)], sag=0.4)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
    ax[0].plot(x, -y, color=CORAL, lw=3); ax[0].set_title("hanging: almost a V")
    ax[1].plot(x, y, color=TEAL, lw=3); ax[1].set_title("upside down: a pointed arch")
    for a in ax: a.set_aspect("equal")
    plt.show()
    print("Two nearly straight lines meeting at the weight (exactly straight if the chain weighed nothing).")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
Galileo, *Two New Sciences* (1638). Hooke, *A Description of Helioscopes* (1675). Wikipedia: Catenary; Church of
Colònia Güell. J. Heyman, *The Stone Skeleton* (1995). M. Milanković (1907), via Heyman. All retrieved 2026-10-30.